In [1]:
import warnings
warnings.filterwarnings('ignore')
from network_analysis import NetworkAnalyzer

# 초기화
analyzer = NetworkAnalyzer()
results_summary = {}
save_path = '../result/'
data_path = '../db/processed_data/total_only.csv'

# 데이터 로드 및 준비
total_df_only = analyzer.preprocessor.load_and_prepare_data(data_path)
datasets = analyzer.preprocessor.create_group_datasets(total_df_only)

print("=== 한국인 식습관-건강 네트워크 분석 ===")
print(f"분석 대상: {len(total_df_only):,}명")
print(f"- 남성: {len(datasets['Men_All']):,}명")
print(f"- 여성: {len(datasets['Women_All']):,}명")
print(f"- 식품군: 12개, 건강지표: 23개\n")

# 기본 네트워크 생성
basic_networks = analyzer.create_basic_networks(datasets)
centrality_results = analyzer.centrality_analyzer.analyze_network_centrality_from_graphs(basic_networks)

# 그룹 분리
age_groups = ['under40', '40-49', '50-64', '65plus']
age_labels = ['40세 미만', '40-49세', '50-64세', '65세 이상']
health_types = ['diet_disease', 'diet_mets', 'diet_biomarker']
health_names = ['질병', 'MetS', '생체지표']

=== 한국인 식습관-건강 네트워크 분석 ===
분석 대상: 23,040명
- 남성: 12,294명
- 여성: 10,746명
- 식품군: 12개, 건강지표: 23개

Analyzing Total - Co-occurrence (poor)
Disconnected graph
Analyzing Total - Co-occurrence (non_poor)
Disconnected graph
Analyzing Total - Health Network (diet_disease)
Connected graph
Analyzing Total - Health Network (diet_mets)
Connected graph
Analyzing Total - Health Network (diet_biomarker)
Connected graph
Analyzing Men_All - Co-occurrence (poor)
Disconnected graph
Analyzing Men_All - Co-occurrence (non_poor)
Disconnected graph
Analyzing Men_All - Health Network (diet_disease)
Connected graph
Analyzing Men_All - Health Network (diet_mets)
Connected graph
Analyzing Men_All - Health Network (diet_biomarker)
Disconnected graph
Analyzing Men_under40 - Co-occurrence (poor)
Disconnected graph
Analyzing Men_under40 - Co-occurrence (non_poor)
Disconnected graph
Analyzing Men_under40 - Health Network (diet_disease)
Disconnected graph
Analyzing Men_under40 - Health Network (diet_mets)
Disconnected gra

# 1 식품군 섭취 패턴 네트워크

## 1.1 전체 인구집단 분석

### 식품군 간 섭취 상관관계 (Spearman)

In [2]:
# 상관관계 분석
diet_corr_patterns = {}
for group_name, data in datasets.items():
    if len(data) >= 100:
        patterns_df, corr_matrix = analyzer.correlation_network.analyze_diet_correlation_patterns(data)
        diet_corr_patterns[group_name] = {'patterns': patterns_df, 'matrix': corr_matrix}

# 전체 그룹 결과 출력
total_patterns = diet_corr_patterns['Total']['patterns']
print("\n◆ 정적 상관관계 (r>0.3):")
positive = total_patterns[total_patterns['Pattern_Type'] == 'Positive']
if not positive.empty:
    for _, row in positive.iterrows():
        print(f"  - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")
else:
    print("  - 없음")

print("\n◆ 부적 상관관계 (r<-0.3):")
negative = total_patterns[total_patterns['Pattern_Type'] == 'Negative']
if not negative.empty:
    for _, row in negative.iterrows():
        print(f"  - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")
else:
    print("  - 없음")

print("\n◆ 독립적 관계 (-0.1<r<0.1):")
independent = total_patterns[total_patterns['Pattern_Type'] == 'Independent']
print(f"  - {len(independent)}개 식품군 쌍이 독립적 섭취 패턴")


◆ 정적 상관관계 (r>0.3):
  - Protein ↔ Vegetables: r=0.475
  - Fried ↔ High Fat
Meat: r=0.363
  - Fried ↔ Processed
Foods: r=0.334
  - Processed
Foods ↔ SSB: r=0.334
  - Salty
Food ↔ Add-Salt: r=0.350

◆ 부적 상관관계 (r<-0.3):
  - 없음

◆ 독립적 관계 (-0.1<r<0.1):
  - 31개 식품군 쌍이 독립적 섭취 패턴


### 네트워크 중심성 분석

In [3]:
print("\n3.1.1 식품군 중심성 분석")
print("-" * 40)

# Poor/Non-Poor Diet 허브 분석
for quality in ['poor', 'non_poor']:
    quality_title = 'Poor' if quality == 'poor' else 'Non-Poor'
    print(f"\n◆ {quality_title} Diet 허브:")
    
    centrality_data = centrality_results['Total']['cooccurrence'].get(quality, {})
    degree_sorted = sorted(centrality_data['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:5]
    print("  Degree Centrality 상위 5개:")
    for i, (node, score) in enumerate(degree_sorted):
        print(f"    {i+1}. {node}: {score:.3f}")
    
    between_sorted = sorted(centrality_data['betweenness_centrality'].items(), key=lambda x: x[1], reverse=True)[:3]
    if between_sorted and between_sorted[0][1] > 0:
        print("  브릿지 식품군 (Betweenness):")
        for node, score in between_sorted:
            if score > 0:
                print(f"    • {node}: {score:.3f}")


3.1.1 식품군 중심성 분석
----------------------------------------

◆ Poor Diet 허브:
  Degree Centrality 상위 5개:
    1. Dairy: 0.545
    2. Vegetables: 0.455
    3. Fruits: 0.455
    4. Grain: 0.364
    5. Protein: 0.364
  브릿지 식품군 (Betweenness):
    • Dairy: 0.103
    • Vegetables: 0.012
    • Fruits: 0.012

◆ Non-Poor Diet 허브:
  Degree Centrality 상위 5개:
    1. Fried: 0.455
    2. High Fat
Meat: 0.455
    3. Processed
Foods: 0.455
    4. Add-Salt: 0.455
    5. Salty
Food: 0.364
  브릿지 식품군 (Betweenness):
    • Fried: 0.005
    • High Fat
Meat: 0.005
    • Processed
Foods: 0.005


In [4]:
print("\n식품군 섭취 질 네트워크 기본 구조")
print("-" * 40)

# Poor/Non-Poor 동시발생 분석
cooccur_stats = {}
for quality in ['poor', 'non_poor']:
    G = basic_networks['Total']['cooccurrence'][quality]['graph']
    matrix = basic_networks['Total']['cooccurrence'][quality]['matrix']
    
    node_counts = {node: G.nodes[node]['count'] for node in G.nodes()}
    if G.number_of_edges() > 0:
        edge_counts = [(u, v, G.edges[u,v]['count']) for u, v in G.edges()]
        edge_counts.sort(key=lambda x: x[2], reverse=True)
    else:
        edge_counts = []
    
    cooccur_stats[quality] = {
        'nodes': node_counts,
        'edges': edge_counts[:5],  # 상위 5개만
        'n_edges': G.number_of_edges()
    }

quality_title_map = {'poor': 'Poor Diet Co-occurrence (1점)', 'non_poor': 'Non-Poor Diet Co-occurrence (3-5점)'}

for quality in ['poor', 'non_poor']:
    print(f"\n◆ {quality_title_map[quality]}:")
    print(f"  - 네트워크 엣지 수: {cooccur_stats[quality]['n_edges']}")
    if cooccur_stats[quality]['edges']:
        print("  - 상위 동시발생 패턴:")
        for u, v, count in cooccur_stats[quality]['edges'][:3]:
            print(f"    • {u} + {v}: {count}명")


식품군 섭취 질 네트워크 기본 구조
----------------------------------------

◆ Poor Diet Co-occurrence (1점):
  - 네트워크 엣지 수: 14
  - 상위 동시발생 패턴:
    • Protein + Vegetables: 4147명
    • Vegetables + Dairy: 3912명
    • Fruits + Dairy: 3691명

◆ Non-Poor Diet Co-occurrence (3-5점):
  - 네트워크 엣지 수: 14
  - 상위 동시발생 패턴:
    • Fried + High Fat
Meat: 20854명
    • Fried + Add-Salt: 20779명
    • High Fat
Meat + Add-Salt: 20621명


## 1.2 성별 간 분석

In [5]:
print("성별 간 식습관 네트워크 차이")
print("-" * 40)

# 성별 간 식습관 상관관계 패턴 비교
print("\n◆ 성별 식습관 상관관계 차이:")
for quality in ['poor', 'non_poor']:
    quality_title = 'Poor' if quality == 'poor' else 'Non-Poor'
    
    # 남녀별 cooccurrence 네트워크 속성 비교
    men_props = centrality_results['Men_All']['cooccurrence'].get(quality, {}).get('network_properties', {})
    women_props = centrality_results['Women_All']['cooccurrence'].get(quality, {}).get('network_properties', {})
    
    print(f"\n  {quality_title} Diet:")
    print(f"    밀도: 남성={men_props.get('density', 0):.3f}, 여성={women_props.get('density', 0):.3f}")
    print(f"    연결수: 남성={men_props.get('edges', 0)}, 여성={women_props.get('edges', 0)}")

# 성별 주요 허브 식품군 비교 (Poor/Non-Poor Diet)
print("\n◆ 성별 주요 허브 식품군:")
for quality in ['poor', 'non_poor']:
    quality_title = 'Poor' if quality == 'poor' else 'Non-Poor'
    print(f"\n  {quality_title} Diet 허브:")
    
    for gender, gender_name in [('Men_All', '남성'), ('Women_All', '여성')]:
        centrality_data = centrality_results[gender]['cooccurrence'].get(quality, {})
        if 'degree_centrality' in centrality_data:
            top_hubs = sorted(centrality_data['degree_centrality'].items(), 
                            key=lambda x: x[1], reverse=True)[:3]
            hub_list = [f"{node}({score:.2f})" for node, score in top_hubs]
            print(f"    {gender_name}: {', '.join(hub_list)}")

# 식습관 상관관계 패턴의 성별 차이
print("\n◆ 식습관 상관관계 강도 비교:")
men_patterns = diet_corr_patterns['Men_All']['patterns']
women_patterns = diet_corr_patterns['Women_All']['patterns']

# 강한 상관관계 (|r| > 0.3) 개수 비교
men_strong = len(men_patterns[abs(men_patterns['Correlation']) > 0.3])
women_strong = len(women_patterns[abs(women_patterns['Correlation']) > 0.3])
print(f"  강한 상관관계(|r|>0.3): 남성={men_strong}개, 여성={women_strong}개")

# 독립적 관계 (-0.1 < r < 0.1) 개수 비교  
men_independent = len(men_patterns[men_patterns['Pattern_Type'] == 'Independent'])
women_independent = len(women_patterns[women_patterns['Pattern_Type'] == 'Independent'])
print(f"  독립적 관계: 남성={men_independent}개, 여성={women_independent}개")

성별 간 식습관 네트워크 차이
----------------------------------------

◆ 성별 식습관 상관관계 차이:

  Poor Diet:
    밀도: 남성=0.212, 여성=0.212
    연결수: 남성=14, 여성=14

  Non-Poor Diet:
    밀도: 남성=0.212, 여성=0.212
    연결수: 남성=14, 여성=14

◆ 성별 주요 허브 식품군:

  Poor Diet 허브:
    남성: Dairy(0.64), Vegetables(0.45), Grain(0.36)
    여성: Dairy(0.55), Protein(0.45), Vegetables(0.45)

  Non-Poor Diet 허브:
    남성: Add-Salt(0.55), Fried(0.45), Processed
Foods(0.45)
    여성: Fried(0.45), High Fat
Meat(0.45), SSB(0.45)

◆ 식습관 상관관계 강도 비교:
  강한 상관관계(|r|>0.3): 남성=7개, 여성=4개
  독립적 관계: 남성=31개, 여성=32개


## 1.3. 성별/연령별 분석

In [6]:
print("\n◆ 연령별 Poor Diet 주요 허브 식품군:")
age_hubs = {}

for gender in ['Men', 'Women']:
   print(f"\n  {gender}성:")
   gender_hubs = {}
   
   for age, label in zip(age_groups, age_labels):
       key = f"{gender}_{age}"
       if key in centrality_results:
           poor_data = centrality_results[key]['cooccurrence'].get('poor', {})
           if 'degree_centrality' in poor_data and poor_data['degree_centrality']:
               top_hubs = sorted(poor_data['degree_centrality'].items(), 
                               key=lambda x: x[1], reverse=True)[:3]
               hub_list = [f"{node}({score:.2f})" for node, score in top_hubs]
               gender_hubs[age] = top_hubs
               print(f"    {label}: {', '.join(hub_list)}")
           else:
               gender_hubs[age] = []
               print(f"    {label}: 데이터 부족")
       else:
           gender_hubs[age] = []
           print(f"    {label}: 분석 대상 없음")
   
   age_hubs[gender] = gender_hubs

print("\n◆ 연령별 Non-Poor Diet 주요 허브 식품군:")
non_poor_age_hubs = {}

for gender in ['Men', 'Women']:
   print(f"\n  {gender}성:")
   non_poor_gender_hubs = {}
   
   for age, label in zip(age_groups, age_labels):
       key = f"{gender}_{age}"
       if key in centrality_results:
           non_poor_data = centrality_results[key]['cooccurrence'].get('non_poor', {})
           if 'degree_centrality' in non_poor_data and non_poor_data['degree_centrality']:
               top_hubs = sorted(non_poor_data['degree_centrality'].items(), 
                               key=lambda x: x[1], reverse=True)[:3]
               hub_list = [f"{node}({score:.2f})" for node, score in top_hubs]
               non_poor_gender_hubs[age] = top_hubs
               print(f"    {label}: {', '.join(hub_list)}")
           else:
               non_poor_gender_hubs[age] = []
               print(f"    {label}: 데이터 부족")
       else:
           non_poor_gender_hubs[age] = []
           print(f"    {label}: 분석 대상 없음")
   
   non_poor_age_hubs[gender] = non_poor_gender_hubs


◆ 연령별 Poor Diet 주요 허브 식품군:

  Men성:
    40세 미만: Fruits(0.64), Vegetables(0.45), Dairy(0.36)
    40-49세: Dairy(0.64), Vegetables(0.45), Grain(0.36)
    50-64세: Dairy(0.55), Vegetables(0.45), Fruits(0.45)
    65세 이상: Vegetables(0.55), Dairy(0.55), Protein(0.36)

  Women성:
    40세 미만: Vegetables(0.55), Sweet
Food(0.55), Protein(0.36)
    40-49세: Protein(0.45), Vegetables(0.45), Dairy(0.45)
    50-64세: Protein(0.55), Dairy(0.55), Vegetables(0.45)
    65세 이상: Vegetables(0.55), Dairy(0.55), Protein(0.45)

◆ 연령별 Non-Poor Diet 주요 허브 식품군:

  Men성:
    40세 미만: Add-Salt(0.55), Grain(0.45), Fried(0.45)
    40-49세: Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)
    50-64세: Add-Salt(0.55), Fried(0.45), Processed
Foods(0.45)
    65세 이상: Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)

  Women성:
    40세 미만: Fried(0.55), High Fat
Meat(0.55), Add-Salt(0.45)
    40-49세: Fried(0.45), High Fat
Meat(0.45), SSB(0.45)
    50-64세: Fried(0.45), High Fat
Meat(0.45), SSB(0.45)
    65세 이상: Fried(

# 3. 결과 저장

In [7]:
results_summary['diet_correlation'] = diet_corr_patterns
results_summary['cooccurrence_stats'] = cooccur_stats

In [8]:
# 네트워크 시각화
analyzer.create_optimized_plots(basic_networks, centrality_results, save_path)

Created Men diet age progression plot
Created Women diet age progression plot
Created Men health-diet age progression plot
Created Women health-diet age progression plot


# 4. MetS 유무 비교 분석 (DM/AMI/Stroke/CKD 제외)

## 4.1 MetS 그룹 생성

In [9]:
# MetS 비교를 위한 데이터셋 생성 (DM, AMI, Stroke, CKD 제외)
mets_datasets, filtered_data = analyzer.preprocessor.create_mets_comparison_datasets(total_df_only)

print("=== MetS 유무 비교 분석 (DM/AMI/Stroke/CKD 제외) ===")
print(f"필터링 후 대상자: {len(filtered_data):,}명")
print(f"- MetS(+): {len(mets_datasets['MetS_Positive_Total']):,}명 ({len(mets_datasets['MetS_Positive_Total'])/len(filtered_data)*100:.1f}%)")
print(f"  • 남성: {len(mets_datasets['MetS_Positive_Men']):,}명 ({len(mets_datasets['MetS_Positive_Men'])/len(mets_datasets['MetS_Positive_Total'])*100:.1f}%)")
print(f"  • 여성: {len(mets_datasets['MetS_Positive_Women']):,}명 ({len(mets_datasets['MetS_Positive_Women'])/len(mets_datasets['MetS_Positive_Total'])*100:.1f}%)")
print(f"- MetS(-): {len(mets_datasets['MetS_Negative_Total']):,}명 ({len(mets_datasets['MetS_Negative_Total'])/len(filtered_data)*100:.1f}%)")
print(f"  • 남성: {len(mets_datasets['MetS_Negative_Men']):,}명 ({len(mets_datasets['MetS_Negative_Men'])/len(mets_datasets['MetS_Negative_Total'])*100:.1f}%)")
print(f"  • 여성: {len(mets_datasets['MetS_Negative_Women']):,}명 ({len(mets_datasets['MetS_Negative_Women'])/len(mets_datasets['MetS_Negative_Total'])*100:.1f}%)\n")

# MetS 그룹별 네트워크 생성
mets_networks = analyzer.create_basic_networks(mets_datasets)
mets_centrality = analyzer.centrality_analyzer.analyze_network_centrality_from_graphs(mets_networks)

=== MetS 유무 비교 분석 (DM/AMI/Stroke/CKD 제외) ===
필터링 후 대상자: 20,949명
- MetS(+): 4,637명 (22.1%)
  • 남성: 3,437명 (74.1%)
  • 여성: 1,200명 (25.9%)
- MetS(-): 16,312명 (77.9%)
  • 남성: 7,293명 (44.7%)
  • 여성: 9,019명 (55.3%)

Analyzing MetS_Positive_Total - Co-occurrence (poor)
Disconnected graph
Analyzing MetS_Positive_Total - Co-occurrence (non_poor)
Disconnected graph
Analyzing MetS_Positive_Total - Health Network (diet_disease)
Disconnected graph
Analyzing MetS_Positive_Total - Health Network (diet_mets)
Disconnected graph
Analyzing MetS_Positive_Total - Health Network (diet_biomarker)
Disconnected graph
Analyzing MetS_Negative_Total - Co-occurrence (poor)
Disconnected graph
Analyzing MetS_Negative_Total - Co-occurrence (non_poor)
Disconnected graph
Analyzing MetS_Negative_Total - Health Network (diet_disease)
Disconnected graph
Analyzing MetS_Negative_Total - Health Network (diet_mets)
Connected graph
Analyzing MetS_Negative_Total - Health Network (diet_biomarker)
Disconnected graph
Analyzing Met

## 4.2 MetS 그룹 간 식품군 섭취 상관관계 비교

In [10]:
# MetS 그룹별 식품군 상관관계 분석
mets_corr_patterns = {}
for group_name, data in mets_datasets.items():
    if len(data) >= 100:
        patterns_df, corr_matrix = analyzer.correlation_network.analyze_diet_correlation_patterns(data)
        mets_corr_patterns[group_name] = {'patterns': patterns_df, 'matrix': corr_matrix}

print("◆ MetS 유무에 따른 식품군 섭취 상관관계 패턴 비교")
print("-" * 60)

# 전체 그룹 비교
pos_patterns = mets_corr_patterns['MetS_Positive_Total']['patterns']
neg_patterns = mets_corr_patterns['MetS_Negative_Total']['patterns']

print("\n[전체 대상자]")
print("\n✓ MetS(+) 그룹:")
print(f"  • 강한 정적 상관(r>0.3): {len(pos_patterns[pos_patterns['Correlation'] > 0.3])}개")
pos_strong_positive = pos_patterns[pos_patterns['Correlation'] > 0.3].sort_values('Correlation', ascending=False)
if len(pos_strong_positive) > 0:
    for _, row in pos_strong_positive.head(3).iterrows():
        print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

print(f"\n  • 강한 부적 상관(r<-0.3): {len(pos_patterns[pos_patterns['Correlation'] < -0.3])}개")
pos_strong_negative = pos_patterns[pos_patterns['Correlation'] < -0.3].sort_values('Correlation')
if len(pos_strong_negative) > 0:
    for _, row in pos_strong_negative.head(3).iterrows():
        print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

print("\n✓ MetS(-) 그룹:")
print(f"  • 강한 정적 상관(r>0.3): {len(neg_patterns[neg_patterns['Correlation'] > 0.3])}개")
neg_strong_positive = neg_patterns[neg_patterns['Correlation'] > 0.3].sort_values('Correlation', ascending=False)
if len(neg_strong_positive) > 0:
    for _, row in neg_strong_positive.head(3).iterrows():
        print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

print(f"\n  • 강한 부적 상관(r<-0.3): {len(neg_patterns[neg_patterns['Correlation'] < -0.3])}개")
neg_strong_negative = neg_patterns[neg_patterns['Correlation'] < -0.3].sort_values('Correlation')
if len(neg_strong_negative) > 0:
    for _, row in neg_strong_negative.head(3).iterrows():
        print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

# 성별 비교
print("\n\n[성별 비교]")
for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n✓ {gender_name}:")
    pos_key = f'MetS_Positive_{gender}'
    neg_key = f'MetS_Negative_{gender}'
    
    if pos_key in mets_corr_patterns and neg_key in mets_corr_patterns:
        pos_g = mets_corr_patterns[pos_key]['patterns']
        neg_g = mets_corr_patterns[neg_key]['patterns']
        
        pos_strong = len(pos_g[abs(pos_g['Correlation']) > 0.3])
        neg_strong = len(neg_g[abs(neg_g['Correlation']) > 0.3])
        
        print(f"  강한 상관관계(|r|>0.3) 개수: MetS(+)={pos_strong}개, MetS(-)={neg_strong}개")

◆ MetS 유무에 따른 식품군 섭취 상관관계 패턴 비교
------------------------------------------------------------

[전체 대상자]

✓ MetS(+) 그룹:
  • 강한 정적 상관(r>0.3): 6개
    - Protein ↔ Vegetables: r=0.438
    - Fried ↔ High Fat
Meat: r=0.378
    - Fried ↔ Processed
Foods: r=0.373

  • 강한 부적 상관(r<-0.3): 0개

✓ MetS(-) 그룹:
  • 강한 정적 상관(r>0.3): 5개
    - Protein ↔ Vegetables: r=0.492
    - Fried ↔ High Fat
Meat: r=0.362
    - Salty
Food ↔ Add-Salt: r=0.344

  • 강한 부적 상관(r<-0.3): 0개


[성별 비교]

✓ 남성:
  강한 상관관계(|r|>0.3) 개수: MetS(+)=5개, MetS(-)=6개

✓ 여성:
  강한 상관관계(|r|>0.3) 개수: MetS(+)=4개, MetS(-)=3개


## 4.3 MetS 유무에 따른 식습관 동시발생 패턴 비교

In [11]:
print("◆ MetS 유무에 따른 Poor/Non-Poor Diet 동시발생 패턴")
print("-" * 60)

# Poor/Non-Poor Diet 동시발생 비교
mets_cooccur_stats = {}
for group_key in ['MetS_Positive_Total', 'MetS_Negative_Total', 
                  'MetS_Positive_Men', 'MetS_Negative_Men',
                  'MetS_Positive_Women', 'MetS_Negative_Women']:
    if group_key in mets_networks:
        mets_cooccur_stats[group_key] = {}
        for quality in ['poor', 'non_poor']:
            G = mets_networks[group_key]['cooccurrence'][quality]['graph']
            matrix = mets_networks[group_key]['cooccurrence'][quality]['matrix']
            
            if G.number_of_edges() > 0:
                edge_counts = [(u, v, G.edges[u,v]['count']) for u, v in G.edges()]
                edge_counts.sort(key=lambda x: x[2], reverse=True)
            else:
                edge_counts = []
            
            mets_cooccur_stats[group_key][quality] = {
                'n_edges': G.number_of_edges(),
                'edges': edge_counts[:5],
                'total_cooccurrences': sum([e[2] for e in edge_counts])
            }

# 전체 비교
print("\n[전체 대상자]")
for quality, quality_name in [('poor', 'Poor Diet (1점)'), ('non_poor', 'Non-Poor Diet (3-5점)')]:
    print(f"\n✓ {quality_name}:")
    
    pos_stats = mets_cooccur_stats['MetS_Positive_Total'][quality]
    neg_stats = mets_cooccur_stats['MetS_Negative_Total'][quality]
    
    print(f"  MetS(+): 엣지 {pos_stats['n_edges']}개, 총 동시발생 {pos_stats['total_cooccurrences']:,}명")
    if pos_stats['edges']:
        print("    주요 패턴:")
        for u, v, count in pos_stats['edges'][:3]:
            print(f"      • {u} + {v}: {count:,}명")
    
    print(f"\n  MetS(-): 엣지 {neg_stats['n_edges']}개, 총 동시발생 {neg_stats['total_cooccurrences']:,}명")
    if neg_stats['edges']:
        print("    주요 패턴:")
        for u, v, count in neg_stats['edges'][:3]:
            print(f"      • {u} + {v}: {count:,}명")

# 성별 비교
print("\n\n[성별 비교]")
for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n✓ {gender_name}:")
    pos_key = f'MetS_Positive_{gender}'
    neg_key = f'MetS_Negative_{gender}'
    
    for quality, quality_name in [('poor', 'Poor'), ('non_poor', 'Non-Poor')]:
        pos_stats = mets_cooccur_stats[pos_key][quality]
        neg_stats = mets_cooccur_stats[neg_key][quality]
        
        print(f"  {quality_name} Diet: MetS(+) {pos_stats['n_edges']}개 엣지, MetS(-) {neg_stats['n_edges']}개 엣지")

◆ MetS 유무에 따른 Poor/Non-Poor Diet 동시발생 패턴
------------------------------------------------------------

[전체 대상자]

✓ Poor Diet (1점):
  MetS(+): 엣지 14개, 총 동시발생 6,559명
    주요 패턴:
      • Fruits + Dairy: 849명
      • Vegetables + Dairy: 848명
      • Protein + Vegetables: 791명

  MetS(-): 엣지 14개, 총 동시발생 22,310명
    주요 패턴:
      • Protein + Vegetables: 2,998명
      • Vegetables + Dairy: 2,734명
      • Fruits + Dairy: 2,562명

✓ Non-Poor Diet (3-5점):
  MetS(+): 엣지 14개, 총 동시발생 55,652명
    주요 패턴:
      • Fried + Add-Salt: 4,167명
      • Fried + High Fat
Meat: 4,146명
      • Fried + Processed
Foods: 4,112명

  MetS(-): 엣지 14개, 총 동시발생 199,019명
    주요 패턴:
      • Fried + High Fat
Meat: 14,773명
      • Fried + Add-Salt: 14,703명
      • High Fat
Meat + Add-Salt: 14,612명


[성별 비교]

✓ 남성:
  Poor Diet: MetS(+) 14개 엣지, MetS(-) 14개 엣지
  Non-Poor Diet: MetS(+) 14개 엣지, MetS(-) 14개 엣지

✓ 여성:
  Poor Diet: MetS(+) 14개 엣지, MetS(-) 14개 엣지
  Non-Poor Diet: MetS(+) 14개 엣지, MetS(-) 14개 엣지


## 4.4 MetS 유무에 따른 네트워크 중심성 비교

In [12]:
print("◆ MetS(+) vs MetS(-): 네트워크 밀도 및 주요 허브 비교")
print("-" * 60)

print("\n[네트워크 밀도 비교]")
for quality, quality_name in [('poor', 'Poor Diet'), ('non_poor', 'Non-Poor Diet')]:
    print(f"\n✓ {quality_name}:")
    
    pos_cooccur = mets_centrality['MetS_Positive_Total']['cooccurrence'][quality]
    neg_cooccur = mets_centrality['MetS_Negative_Total']['cooccurrence'][quality]
    
    pos_density = pos_cooccur['network_properties'].get('density', 0)
    neg_density = neg_cooccur['network_properties'].get('density', 0)
    
    print(f"  전체: MetS(+)={pos_density:.3f}, MetS(-)={neg_density:.3f}")
    
    # 성별 밀도
    for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
        pos_key = f'MetS_Positive_{gender}'
        neg_key = f'MetS_Negative_{gender}'
        
        pos_d = mets_centrality[pos_key]['cooccurrence'][quality]['network_properties'].get('density', 0)
        neg_d = mets_centrality[neg_key]['cooccurrence'][quality]['network_properties'].get('density', 0)
        
        print(f"  {gender_name}: MetS(+)={pos_d:.3f}, MetS(-)={neg_d:.3f}")

print("\n\n[주요 허브 식품군 비교]")
for quality, quality_name in [('poor', 'Poor Diet'), ('non_poor', 'Non-Poor Diet')]:
    print(f"\n✓ {quality_name}:")
    
    # 전체
    pos_cooccur = mets_centrality['MetS_Positive_Total']['cooccurrence'][quality]
    neg_cooccur = mets_centrality['MetS_Negative_Total']['cooccurrence'][quality]
    
    pos_top3 = sorted(pos_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:3]
    neg_top3 = sorted(neg_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:3]
    
    print(f"  전체 - MetS(+): {', '.join([f'{n}({s:.2f})' for n, s in pos_top3])}")
    print(f"  전체 - MetS(-): {', '.join([f'{n}({s:.2f})' for n, s in neg_top3])}")
    
    # 성별
    for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
        pos_key = f'MetS_Positive_{gender}'
        neg_key = f'MetS_Negative_{gender}'
        
        pos_cooccur = mets_centrality[pos_key]['cooccurrence'][quality]
        neg_cooccur = mets_centrality[neg_key]['cooccurrence'][quality]
        
        pos_top = sorted(pos_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:2]
        neg_top = sorted(neg_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:2]
        
        print(f"  {gender_name} - MetS(+): {', '.join([f'{n}({s:.2f})' for n, s in pos_top])}")
        print(f"  {gender_name} - MetS(-): {', '.join([f'{n}({s:.2f})' for n, s in neg_top])}")

◆ MetS(+) vs MetS(-): 네트워크 밀도 및 주요 허브 비교
------------------------------------------------------------

[네트워크 밀도 비교]

✓ Poor Diet:
  전체: MetS(+)=0.212, MetS(-)=0.212
  남성: MetS(+)=0.212, MetS(-)=0.212
  여성: MetS(+)=0.212, MetS(-)=0.212

✓ Non-Poor Diet:
  전체: MetS(+)=0.212, MetS(-)=0.212
  남성: MetS(+)=0.212, MetS(-)=0.212
  여성: MetS(+)=0.212, MetS(-)=0.212


[주요 허브 식품군 비교]

✓ Poor Diet:
  전체 - MetS(+): Vegetables(0.55), Dairy(0.55), Grain(0.36)
  전체 - MetS(-): Dairy(0.55), Protein(0.45), Vegetables(0.45)
  남성 - MetS(+): Fruits(0.55), Vegetables(0.45)
  남성 - MetS(-): Dairy(0.55), Vegetables(0.45)
  여성 - MetS(+): Vegetables(0.55), Dairy(0.55)
  여성 - MetS(-): Protein(0.45), Vegetables(0.45)

✓ Non-Poor Diet:
  전체 - MetS(+): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)
  전체 - MetS(-): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)
  남성 - MetS(+): Fried(0.55), Add-Salt(0.55)
  남성 - MetS(-): Fried(0.45), High Fat
Meat(0.45)
  여성 - MetS(+): Fried(0.45), High Fat
Meat(0.45)


## 4.5 MetS와 식습관 질 조합 분석

In [13]:
print("◆ MetS 유무 × 식습관 질 조합 분석")
print("-" * 60)

# 식품군별 Poor/Non-Poor 점수 분포 비교
import pandas as pd
import numpy as np

food_groups = ['Grain', 'Protein', 'Vegetables', 'Fruits', 'Dairy', 
               'Fried', 'Sweet\nFood', 'High Fat\nMeat', 'Processed\nFoods', 
               'SSB', 'Salty\nFood', 'Add-Salt']

print("\n[식품군별 Poor Diet (1점) 비율 비교]")
print("\n전체:")
poor_comparison = {}
for fg in food_groups:
    pos_data = mets_datasets['MetS_Positive_Total']
    neg_data = mets_datasets['MetS_Negative_Total']
    
    if fg in pos_data.columns and fg in neg_data.columns:
        pos_poor_rate = (pos_data[fg] == 1).sum() / len(pos_data) * 100
        neg_poor_rate = (neg_data[fg] == 1).sum() / len(neg_data) * 100
        diff = pos_poor_rate - neg_poor_rate
        
        poor_comparison[fg] = {
            'MetS(+)': pos_poor_rate,
            'MetS(-)': neg_poor_rate,
            'diff': diff
        }

# 차이가 큰 순으로 정렬
sorted_comparison = sorted(poor_comparison.items(), key=lambda x: abs(x[1]['diff']), reverse=True)

for fg, rates in sorted_comparison[:8]:
    print(f"  {fg.replace(chr(10), ' ')}: MetS(+)={rates['MetS(+)']:.1f}%, MetS(-)={rates['MetS(-)']:.1f}%, 차이={rates['diff']:+.1f}%p")

# 성별 분석
print("\n\n[성별 Poor Diet 비율 차이]")
for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n{gender_name} (상위 5개):")
    pos_key = f'MetS_Positive_{gender}'
    neg_key = f'MetS_Negative_{gender}'
    
    gender_comparison = {}
    for fg in food_groups:
        pos_data = mets_datasets[pos_key]
        neg_data = mets_datasets[neg_key]
        
        if fg in pos_data.columns and fg in neg_data.columns:
            pos_poor_rate = (pos_data[fg] == 1).sum() / len(pos_data) * 100
            neg_poor_rate = (neg_data[fg] == 1).sum() / len(neg_data) * 100
            diff = pos_poor_rate - neg_poor_rate
            gender_comparison[fg] = diff
    
    sorted_gender = sorted(gender_comparison.items(), key=lambda x: abs(x[1]), reverse=True)
    for fg, diff in sorted_gender[:5]:
        print(f"  {fg.replace(chr(10), ' ')}: {diff:+.1f}%p")

◆ MetS 유무 × 식습관 질 조합 분석
------------------------------------------------------------

[식품군별 Poor Diet (1점) 비율 비교]

전체:
  Sweet Food: MetS(+)=15.1%, MetS(-)=21.1%, 차이=-6.0%p
  Dairy: MetS(+)=43.5%, MetS(-)=39.7%, 차이=+3.8%p
  Salty Food: MetS(+)=12.9%, MetS(-)=9.4%, 차이=+3.5%p
  Fruits: MetS(+)=29.8%, MetS(-)=27.3%, 차이=+2.5%p
  Grain: MetS(+)=17.2%, MetS(-)=15.4%, 차이=+1.9%p
  SSB: MetS(+)=11.5%, MetS(-)=10.0%, 차이=+1.5%p
  Protein: MetS(+)=23.2%, MetS(-)=24.5%, 차이=-1.2%p
  High Fat Meat: MetS(+)=7.2%, MetS(-)=6.2%, 차이=+1.0%p


[성별 Poor Diet 비율 차이]

남성 (상위 5개):
  Salty Food: +3.0%p
  Sweet Food: -2.3%p
  Vegetables: +2.3%p
  Dairy: +1.9%p
  Fruits: +1.4%p

여성 (상위 5개):
  Sweet Food: -8.0%p
  Fruits: -5.7%p
  Processed Foods: -2.0%p
  Vegetables: -1.4%p
  SSB: -1.3%p


## 4.6 MetS 비교 시각화 및 결과 저장

In [14]:
# MetS 비교 네트워크 시각화
analyzer.create_mets_comparison_plots(mets_networks, mets_centrality, save_path)

# MetS 비교 결과 저장
results_summary['mets_comparison'] = {
    'filtered_data_count': len(filtered_data),
    'mets_positive_count': len(mets_datasets['MetS_Positive_Total']),
    'mets_negative_count': len(mets_datasets['MetS_Negative_Total']),
    'centrality_results': mets_centrality,
    'correlation_patterns': mets_corr_patterns,
    'cooccurrence_stats': mets_cooccur_stats,
    'poor_diet_comparison': poor_comparison
}

print("\n✓ MetS 비교 분석 완료")
print(f"  - 시각화 저장 위치: {save_path}")
print(f"  - 분석 그룹: {len(mets_datasets)}개")
print(f"  - 생성된 네트워크: {sum([len(mets_networks[k]) for k in mets_networks])}개")

Created Figure 1: Poor Diet Co-occurrence Networks
Created Figure 2: Non-Poor Diet Co-occurrence Networks
Created Figure 3: Men Diet Networks
Created Figure 4: Women Diet Networks
Created Figure 5a: Men Age Progression (Poor Diet)
Created Figure 5b: Women Age Progression (Poor Diet)

✓ MetS 비교 분석 완료
  - 시각화 저장 위치: ../result/
  - 분석 그룹: 22개
  - 생성된 네트워크: 44개


## 4.7 MetS 그룹별 식습관 질 종합 점수 비교

In [15]:
print("◆ MetS 그룹별 식습관 질 종합 점수 비교")
print("-" * 60)

# 각 그룹별 Poor Diet 개수 분포
print("\n[Poor Diet 개수 분포 (1점인 식품군 개수)]")

for group_label, pos_key, neg_key in [
    ('전체', 'MetS_Positive_Total', 'MetS_Negative_Total'),
    ('남성', 'MetS_Positive_Men', 'MetS_Negative_Men'),
    ('여성', 'MetS_Positive_Women', 'MetS_Negative_Women')
]:
    pos_data = mets_datasets[pos_key]
    neg_data = mets_datasets[neg_key]
    
    # Poor Diet 개수 계산
    pos_poor_counts = (pos_data[food_groups] == 1).sum(axis=1)
    neg_poor_counts = (neg_data[food_groups] == 1).sum(axis=1)
    
    print(f"\n✓ {group_label}:")
    print(f"  MetS(+): 평균 {pos_poor_counts.mean():.2f}개 (중앙값 {pos_poor_counts.median():.0f}개)")
    print(f"  MetS(-): 평균 {neg_poor_counts.mean():.2f}개 (중앙값 {neg_poor_counts.median():.0f}개)")
    print(f"  차이: {pos_poor_counts.mean() - neg_poor_counts.mean():+.2f}개")
    
    # 분포 비교
    print(f"  분포:")
    for n in [0, 3, 6, 9]:
        pos_pct = (pos_poor_counts >= n).sum() / len(pos_poor_counts) * 100
        neg_pct = (neg_poor_counts >= n).sum() / len(neg_poor_counts) * 100
        print(f"    {n}개 이상: MetS(+)={pos_pct:.1f}%, MetS(-)={neg_pct:.1f}%")

# Non-Poor Diet (3-5점) 개수 분포
print("\n\n[Non-Poor Diet 개수 분포 (3-5점인 식품군 개수)]")

for group_label, pos_key, neg_key in [
    ('전체', 'MetS_Positive_Total', 'MetS_Negative_Total'),
    ('남성', 'MetS_Positive_Men', 'MetS_Negative_Men'),
    ('여성', 'MetS_Positive_Women', 'MetS_Negative_Women')
]:
    pos_data = mets_datasets[pos_key]
    neg_data = mets_datasets[neg_key]
    
    # Non-Poor Diet 개수 계산
    pos_nonpoor_counts = (pos_data[food_groups] >= 3).sum(axis=1)
    neg_nonpoor_counts = (neg_data[food_groups] >= 3).sum(axis=1)
    
    print(f"\n✓ {group_label}:")
    print(f"  MetS(+): 평균 {pos_nonpoor_counts.mean():.2f}개 (중앙값 {pos_nonpoor_counts.median():.0f}개)")
    print(f"  MetS(-): 평균 {neg_nonpoor_counts.mean():.2f}개 (중앙값 {neg_nonpoor_counts.median():.0f}개)")
    print(f"  차이: {pos_nonpoor_counts.mean() - neg_nonpoor_counts.mean():+.2f}개")

◆ MetS 그룹별 식습관 질 종합 점수 비교
------------------------------------------------------------

[Poor Diet 개수 분포 (1점인 식품군 개수)]

✓ 전체:
  MetS(+): 평균 2.16개 (중앙값 2개)
  MetS(-): 평균 2.08개 (중앙값 2개)
  차이: +0.08개
  분포:
    0개 이상: MetS(+)=100.0%, MetS(-)=100.0%
    3개 이상: MetS(+)=37.7%, MetS(-)=35.5%
    6개 이상: MetS(+)=3.2%, MetS(-)=3.2%
    9개 이상: MetS(+)=0.1%, MetS(-)=0.1%

✓ 남성:
  MetS(+): 평균 2.29개 (중앙값 2개)
  MetS(-): 평균 2.23개 (중앙값 2개)
  차이: +0.06개
  분포:
    0개 이상: MetS(+)=100.0%, MetS(-)=100.0%
    3개 이상: MetS(+)=41.0%, MetS(-)=38.4%
    6개 이상: MetS(+)=3.8%, MetS(-)=4.1%
    9개 이상: MetS(+)=0.1%, MetS(-)=0.1%

✓ 여성:
  MetS(+): 평균 1.79개 (중앙값 2개)
  MetS(-): 평균 1.96개 (중앙값 2개)
  차이: -0.18개
  분포:
    0개 이상: MetS(+)=100.0%, MetS(-)=100.0%
    3개 이상: MetS(+)=28.3%, MetS(-)=33.1%
    6개 이상: MetS(+)=1.5%, MetS(-)=2.4%
    9개 이상: MetS(+)=0.0%, MetS(-)=0.1%


[Non-Poor Diet 개수 분포 (3-5점인 식품군 개수)]

✓ 전체:
  MetS(+): 평균 9.84개 (중앙값 10개)
  MetS(-): 평균 9.92개 (중앙값 10개)
  차이: -0.08개

✓ 남성:
  MetS(+): 평균 9.71개 (중앙값 10개)

## 4.8 MetS 유무별 특징적 식습관 패턴 요약

In [16]:
print("=" * 70)
print("◆◆◆ MetS 유무별 식습관 네트워크 특성 종합 요약 ◆◆◆")
print("=" * 70)

print("\n【1. 대상자 특성】")
print(f"  • 총 분석 대상: {len(filtered_data):,}명 (DM/AMI/Stroke/CKD 제외)")
print(f"  • MetS(+): {len(mets_datasets['MetS_Positive_Total']):,}명 ({len(mets_datasets['MetS_Positive_Total'])/len(filtered_data)*100:.1f}%)")
print(f"    - 남성 {len(mets_datasets['MetS_Positive_Men']):,}명, 여성 {len(mets_datasets['MetS_Positive_Women']):,}명")
print(f"  • MetS(-): {len(mets_datasets['MetS_Negative_Total']):,}명 ({len(mets_datasets['MetS_Negative_Total'])/len(filtered_data)*100:.1f}%)")
print(f"    - 남성 {len(mets_datasets['MetS_Negative_Men']):,}명, 여성 {len(mets_datasets['MetS_Negative_Women']):,}명")

print("\n【2. 식습관 상관관계 패턴】")
pos_strong = len(pos_patterns[abs(pos_patterns['Correlation']) > 0.3])
neg_strong = len(neg_patterns[abs(neg_patterns['Correlation']) > 0.3])
print(f"  • 강한 상관관계(|r|>0.3): MetS(+) {pos_strong}개, MetS(-) {neg_strong}개")

print("\n  ✓ MetS(+) 주요 정적 상관:")
pos_top_corr = pos_patterns[pos_patterns['Correlation'] > 0.3].sort_values('Correlation', ascending=False).head(3)
for _, row in pos_top_corr.iterrows():
    print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

print("\n  ✓ MetS(-) 주요 정적 상관:")
neg_top_corr = neg_patterns[neg_patterns['Correlation'] > 0.3].sort_values('Correlation', ascending=False).head(3)
for _, row in neg_top_corr.iterrows():
    print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

print("\n【3. 네트워크 허브 식품군】")
print("\n  ✓ Poor Diet 네트워크:")
pos_poor_hub = sorted(mets_centrality['MetS_Positive_Total']['cooccurrence']['poor']['degree_centrality'].items(), 
                      key=lambda x: x[1], reverse=True)[:3]
neg_poor_hub = sorted(mets_centrality['MetS_Negative_Total']['cooccurrence']['poor']['degree_centrality'].items(), 
                      key=lambda x: x[1], reverse=True)[:3]
print(f"    MetS(+): {', '.join([f'{n}({s:.2f})' for n, s in pos_poor_hub])}")
print(f"    MetS(-): {', '.join([f'{n}({s:.2f})' for n, s in neg_poor_hub])}")

print("\n  ✓ Non-Poor Diet 네트워크:")
pos_nonpoor_hub = sorted(mets_centrality['MetS_Positive_Total']['cooccurrence']['non_poor']['degree_centrality'].items(), 
                         key=lambda x: x[1], reverse=True)[:3]
neg_nonpoor_hub = sorted(mets_centrality['MetS_Negative_Total']['cooccurrence']['non_poor']['degree_centrality'].items(), 
                         key=lambda x: x[1], reverse=True)[:3]
print(f"    MetS(+): {', '.join([f'{n}({s:.2f})' for n, s in pos_nonpoor_hub])}")
print(f"    MetS(-): {', '.join([f'{n}({s:.2f})' for n, s in neg_nonpoor_hub])}")

print("\n【4. Poor Diet 차이가 큰 식품군 (상위 5개)】")
sorted_poor_comp = sorted(poor_comparison.items(), key=lambda x: abs(x[1]['diff']), reverse=True)
for fg, rates in sorted_poor_comp[:5]:
    direction = "높음" if rates['diff'] > 0 else "낮음"
    print(f"  • {fg.replace(chr(10), ' ')}: MetS(+)가 {abs(rates['diff']):.1f}%p {direction}")
    print(f"    (MetS(+): {rates['MetS(+)']:.1f}%, MetS(-): {rates['MetS(-)']:.1f}%)")

print("\n【5. 성별 특이 패턴】")
for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    pos_key = f'MetS_Positive_{gender}'
    neg_key = f'MetS_Negative_{gender}'
    
    print(f"\n  ✓ {gender_name}:")
    
    # Poor Diet 허브
    pos_hub = sorted(mets_centrality[pos_key]['cooccurrence']['poor']['degree_centrality'].items(), 
                    key=lambda x: x[1], reverse=True)[:2]
    neg_hub = sorted(mets_centrality[neg_key]['cooccurrence']['poor']['degree_centrality'].items(), 
                    key=lambda x: x[1], reverse=True)[:2]
    print(f"    Poor Diet 허브: MetS(+) {', '.join([n for n, s in pos_hub])}, "
          f"MetS(-) {', '.join([n for n, s in neg_hub])}")
    
    # 네트워크 밀도
    pos_density = mets_centrality[pos_key]['cooccurrence']['poor']['network_properties'].get('density', 0)
    neg_density = mets_centrality[neg_key]['cooccurrence']['poor']['network_properties'].get('density', 0)
    print(f"    Poor 네트워크 밀도: MetS(+)={pos_density:.3f}, MetS(-)={neg_density:.3f}")

print("\n" + "=" * 70)

◆◆◆ MetS 유무별 식습관 네트워크 특성 종합 요약 ◆◆◆

【1. 대상자 특성】
  • 총 분석 대상: 20,949명 (DM/AMI/Stroke/CKD 제외)
  • MetS(+): 4,637명 (22.1%)
    - 남성 3,437명, 여성 1,200명
  • MetS(-): 16,312명 (77.9%)
    - 남성 7,293명, 여성 9,019명

【2. 식습관 상관관계 패턴】
  • 강한 상관관계(|r|>0.3): MetS(+) 6개, MetS(-) 5개

  ✓ MetS(+) 주요 정적 상관:
    - Protein ↔ Vegetables: r=0.438
    - Fried ↔ High Fat
Meat: r=0.378
    - Fried ↔ Processed
Foods: r=0.373

  ✓ MetS(-) 주요 정적 상관:
    - Protein ↔ Vegetables: r=0.492
    - Fried ↔ High Fat
Meat: r=0.362
    - Salty
Food ↔ Add-Salt: r=0.344

【3. 네트워크 허브 식품군】

  ✓ Poor Diet 네트워크:
    MetS(+): Vegetables(0.55), Dairy(0.55), Grain(0.36)
    MetS(-): Dairy(0.55), Protein(0.45), Vegetables(0.45)

  ✓ Non-Poor Diet 네트워크:
    MetS(+): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)
    MetS(-): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)

【4. Poor Diet 차이가 큰 식품군 (상위 5개)】
  • Sweet Food: MetS(+)가 6.0%p 낮음
    (MetS(+): 15.1%, MetS(-): 21.1%)
  • Dairy: MetS(+)가 3.8%p 높음
    (MetS(+): 43